Part 5

In [ ]:
from transformers import DistilBertForSequenceClassification, AutoTokenizer
import torch

MODEL_PATH = "/content/drive/MyDrive/fairness_models/baseline_model"

model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
import pandas as pd

eval_df = pd.read_csv("/content/drive/MyDrive/eval_subset_20k.csv")

sample_df = eval_df.sample(1000, random_state=42)

texts = sample_df["comment_text"].tolist()
labels = sample_df["label"].values

In [ ]:
import numpy as np

def get_probs(texts):
    probs = []

    for i in range(0, len(texts), 32):
        batch = texts[i:i+32]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        batch_probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
        probs.extend(batch_probs.cpu().numpy())

    return np.array(probs)

eval_probs = get_probs(texts)

KeyboardInterrupt: 

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

X_calib = eval_probs.reshape(-1, 1)
y_calib = labels

class Wrapper:
    def __init__(self, probs):
        self.probs = probs
    def predict_proba(self, X):
        return np.vstack([1 - self.probs, self.probs]).T

wrapper = Wrapper(eval_probs)

calibrator = CalibratedClassifierCV(method='isotonic', cv='prefit')
calibrator.fit(X_calib, y_calib)

print("Calibration done")

In [ ]:
from pipeline import ModerationPipeline

pipeline = ModerationPipeline(model, tokenizer, calibrator)

In [ ]:
results = [pipeline.predict(t) for t in texts]

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

layer_counts = results_df["layer"].value_counts()

print(layer_counts)

In [ ]:
import matplotlib.pyplot as plt

layer_counts.plot(kind="bar")
plt.title("Layer Distribution")
plt.ylabel("Count")
plt.show()

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

auto_mask = results_df["decision"] != "review"

auto_preds = (results_df[auto_mask]["decision"] == "block").astype(int)
auto_labels = labels[auto_mask]

print("F1:", f1_score(auto_labels, auto_preds))
print("Precision:", precision_score(auto_labels, auto_preds))
print("Recall:", recall_score(auto_labels, auto_preds))

In [ ]:
review_mask = results_df["decision"] == "review"

review_labels = labels[review_mask]

print("Review Toxic %:", review_labels.mean())